# MedGemma Weak-Labeling for Dental Radiograph Classification (Colab)

Generates **silver labels** for per-tooth / per-image crops using `google/medgemma-4b-it` from Hugging Face.

Assumes your image dataset lives on **Google Drive** and this notebook runs on a **Colab GPU runtime**
(Runtime -> Change runtime type -> T4 GPU or better).

Outputs are written to CSV (machine-parseable), matching the taxonomy already used by
`train_pathology_classifier.py` (`healthy, caries, deep_caries, periapical_lesion, impacted`),
plus additional independent label tasks (restoration type, missing tooth, calculus, bone loss stage).

**These are weak labels.** Treat `uncertain` and low-confidence justifications as needing human
(dentist/radiologist) review before they go into a training set. See the confidence-review cell
at the end for a suggested triage workflow.

Before running:
1. Accept the license for `google/medgemma-4b-it` on its Hugging Face page (logged in with the account
   whose token you'll use below) — it is a gated model.
2. In the Colab sidebar, open the key icon ("Secrets") and add a secret named `HF_TOKEN` with a Hugging
   Face access token that has read access. Do **not** paste the token directly into a cell.
3. Set `DRIVE_IMAGE_DIR` and `DRIVE_OUTPUT_DIR` below to the paths inside your Drive.

## 0. Setup: install deps, mount Drive, authenticate with Hugging Face

In [ ]:
from pathlib import Path

MODEL_ID = "google/medgemma-4b-it"

# Paths inside your mounted Google Drive. Adjust to match your folder layout, e.g.
# "/content/drive/MyDrive/dental-ai/datasets/dentex/unlabeled".
DRIVE_ROOT = Path("/content/drive/MyDrive/dental-ai")
IMAGE_DIR = DRIVE_ROOT / "datasets/dentex/unlabeled"
OUTPUT_CSV = DRIVE_ROOT / "datasets/dentex/medgemma_labels.csv"

assert IMAGE_DIR.exists(), f"IMAGE_DIR not found: {IMAGE_DIR} (check Drive is mounted and path is correct)"

# Optional cap while testing the pipeline; set to None to run over the full folder.
MAX_IMAGES = 20

DEVICE = "cuda"  # Colab GPU runtime; falls back to "cpu" if no GPU is attached
DTYPE = "bfloat16"  # bfloat16 on GPU; use "float32" on CPU

import torch as _torch

if DEVICE == "cuda" and not _torch.cuda.is_available():
    print("No GPU detected — falling back to CPU. Runtime > Change runtime type > GPU for a real run.")
    DEVICE = "cpu"
    DTYPE = "float32"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

## 1. Config

In [ ]:
from pathlib import Path

MODEL_ID = "google/medgemma-4b-it"

# Directory of images to label. Any flat folder of .png/.jpg crops (per-tooth or full OPG).
IMAGE_DIR = Path("../../../datasets/dentex/unlabeled")

# Where results are written.
OUTPUT_CSV = Path("../../../datasets/dentex/medgemma_labels.csv")

# Optional cap while testing the pipeline; set to None to run over the full folder.
MAX_IMAGES = 20

DEVICE = "cuda"  # "cuda", "mps", or "cpu"
DTYPE = "bfloat16"  # bfloat16 on GPU; use "float32" on CPU

## 2. Fixed taxonomy

Each task is independent (multi-label, not mutually exclusive across tasks) and constrained to a
fixed set of choices — MedGemma is noticeably more reliable with closed label sets than open-ended
description. `restoration_type` and `bone_loss_stage` include `not_applicable` since not every
crop will show a restoration or measurable bone loss.

In [ ]:
TAXONOMY = {
    "caries": {
        "labels": ["healthy", "caries", "deep_caries", "uncertain"],
        "question": "Does this dental radiograph crop show visible caries (tooth decay), and if so how severe?",
    },
    "periapical_lesion": {
        "labels": ["absent", "present", "uncertain"],
        "question": "Is there a visible periapical lesion (radiolucency at the root tip) in this image?",
    },
    "impacted": {
        "labels": ["not_impacted", "impacted", "uncertain"],
        "question": "Does this image show an impacted tooth (failed to erupt / blocked by adjacent tooth or bone)?",
    },
    "restoration_type": {
        "labels": ["none", "filling", "crown", "root_canal", "implant", "bridge", "not_applicable", "uncertain"],
        "question": "What type of dental restoration or prior treatment, if any, is visible on this tooth?",
    },
    "missing_tooth": {
        "labels": ["present", "missing", "uncertain"],
        "question": "Is the tooth expected in this position present, or is it missing (edentulous space)?",
    },
    "calculus": {
        "labels": ["absent", "present", "uncertain"],
        "question": "Is there visible calculus (hardened plaque / tartar deposit) on this tooth?",
    },
    "bone_loss_stage": {
        "labels": ["none", "mild", "moderate", "severe", "not_applicable", "uncertain"],
        "question": "What is the severity of alveolar bone loss around this tooth, if any is visible?",
    },
}

## 3. Prompt template + structured output schema

One shared template parameterized per task. The prompt embeds the exact JSON schema and the closed
label set, and asks for a one-sentence justification (useful for human review) plus a confidence
score MedGemma self-reports (heuristic, not calibrated — use it for triage/sorting, not as ground truth).

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are assisting a dental radiologist with annotating a dataset of dental radiograph crops. "
    "You are not making a clinical diagnosis; you are producing a draft label that a dentist will review. "
    "Always answer using only the allowed label values given to you. If the image is ambiguous, blurry, "
    "or you are not confident, answer 'uncertain' rather than guessing."
)


def build_prompt(task_name: str) -> str:
    task = TAXONOMY[task_name]
    labels = task["labels"]
    schema = {
        "type": "object",
        "properties": {
            "label": {"type": "string", "enum": labels},
            "justification": {"type": "string", "description": "One sentence explaining the finding."},
            "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        },
        "required": ["label", "justification", "confidence"],
    }
    return (
        f"{task['question']}\n\n"
        f"Allowed labels: {', '.join(labels)}.\n\n"
        "Respond with ONLY a single JSON object matching this schema, no markdown fences, no extra text:\n"
        f"{json.dumps(schema)}\n"
    )

## 4. Load MedGemma

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

torch_dtype = getattr(torch, DTYPE)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map=DEVICE,
)
model.eval()
print("MedGemma loaded on", model.device)

## 5. Inference helpers

`classify_image` runs one (image, task) pair through MedGemma using the chat template and does a
best-effort JSON parse of the output, with a fallback that flags the row for manual review if
parsing fails rather than silently dropping it.

In [ ]:
import re

from PIL import Image

MAX_NEW_TOKENS = 200


def _extract_json(text: str) -> dict | None:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def classify_image(image: Image.Image, task_name: str) -> dict:
    prompt = build_prompt(task_name)
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        },
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    generated = output[0][input_len:]
    text = processor.decode(generated, skip_special_tokens=True).strip()

    parsed = _extract_json(text)
    if parsed is None or "label" not in parsed:
        return {
            "label": "parse_error",
            "justification": text[:300],
            "confidence": "low",
        }

    allowed = set(TAXONOMY[task_name]["labels"])
    if parsed["label"] not in allowed:
        parsed["justification"] = f"[label '{parsed['label']}' not in allowed set] " + str(parsed.get("justification", ""))
        parsed["label"] = "uncertain" if "uncertain" in allowed else "parse_error"

    parsed.setdefault("confidence", "low")
    parsed.setdefault("justification", "")
    return parsed

## 6. Batch run over the image directory

Runs every task in `TAXONOMY` against every image and writes one row per image to `OUTPUT_CSV`, with
columns `{task}_label`, `{task}_justification`, `{task}_confidence` per task — this is the flat,
machine-parseable format referenced in the notebook intro, easy to filter/pivot into COCO/YOLO/folder
layouts downstream.

In [ ]:
import shutil

STAGE2A_OUT = DRIVE_ROOT / "datasets/dentex/stage2a_medgemma"

for _, row in auto_accept.iterrows():
    label = row["caries_label"]
    if label == "uncertain":
        continue
    dest_dir = STAGE2A_OUT / "train" / label
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(IMAGE_DIR / row["filename"], dest_dir / row["filename"])

print(f"Exported to {STAGE2A_OUT}")

## 7. Confidence triage

These are silver labels. Split into auto-accept vs needs-human-review before using them for
training, rather than trusting every row equally.

In [ ]:
label_cols = [c for c in df.columns if c.endswith("_label")]
conf_cols = [c for c in df.columns if c.endswith("_confidence")]

needs_review_mask = pd.Series(False, index=df.index)
for label_col, conf_col in zip(label_cols, conf_cols):
    needs_review_mask |= df[label_col].isin(["uncertain", "parse_error"])
    needs_review_mask |= df[conf_col].eq("low")

needs_review = df[needs_review_mask]
auto_accept = df[~needs_review_mask]

print(f"Auto-accept: {len(auto_accept)} / {len(df)} ({len(auto_accept) / max(len(df), 1):.1%})")
print(f"Needs human review: {len(needs_review)} / {len(df)}")

review_path = OUTPUT_CSV.with_name(OUTPUT_CSV.stem + "_needs_review.csv")
needs_review.to_csv(review_path, index=False)
print(f"Wrote review queue to {review_path}")

## 8. Optional: export `caries` task to the Stage 2A folder layout

Matches the class-per-folder structure expected by `train_pathology_classifier.py`
(`healthy, caries, deep_caries, periapical_lesion, impacted`). Adjust the mapping if you want to
fold `periapical_lesion`/`impacted` in from their own task columns instead of just `caries`.

In [ ]:
import shutil

STAGE2A_OUT = Path("../../../datasets/dentex/stage2a_medgemma")

for _, row in auto_accept.iterrows():
    label = row["caries_label"]
    if label == "uncertain":
        continue
    dest_dir = STAGE2A_OUT / "train" / label
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(IMAGE_DIR / row["filename"], dest_dir / row["filename"])

print(f"Exported to {STAGE2A_OUT}")